In [51]:
import torch
import torch.nn as nn

In [52]:
class PatchTST(nn.Module):
    def __init__(self, n_channels=6,patch=16,d=64,nhead=4,layers=2,n_class=2):
        super().__init__()
        self.patch=patch
        self.proj=nn.Linear(n_channels*patch,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=nhead,dim_feedforward=128,batch_first=True)
        self.enc=nn.TransformerEncoder(enc,num_layers=layers)

    def feedforward(self,x):
        B,T,C=x.shape
        n=T//self.patch
        x=x[:,:n*self.patch,:]
        x=x.reshape(B,n,C*self.patch)
        h=self.proj(x)
        h=self.enc(h).mean(dim=1)
        return self.head(h)

In [53]:
model=PatchTST()
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
cri=nn.CrossEntropyLoss()

model

PatchTST(
  (proj): Linear(in_features=96, out_features=64, bias=True)
  (enc): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

In [54]:
from pathlib import Path
ROOT=Path.cwd().parent/"dataset"/"imu_data"/"IMU"
FS=128
OBS_S=5.0
HORIZON_S=2.0
STRIDE_S=1.0
FOG_FRAC=0.30
BATCH=32
EPOCHS=25
LR=1e-3
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [55]:
COLS=["acc_ml_g","acc_ap_g","acc_si_g","gyr_ml_dps","gyr_ap_dps","gyr_si_dps"]

In [56]:
import pandas as pd
def read_trial(path):
    path=Path(path)
    if path.suffix.lower()==".txt":
        df=pd.read_csv(path,sep="\t")
        rename={
            "ACC ML [g]": "acc_ml_g",
            "ACC AP [g]": "acc_ap_g",
            "ACC SI [g]": "acc_si_g",
            "GYR ML [deg/s]": "gyr_ml_dps",
            "GYR AP [deg/s]": "gyr_ap_dps",
            "GYR SI [deg/s]": "gyr_si_dps",
            "Freezing event [flag]": "fog_flag",
            "Time [s]": "time_s",
        }
        df=df.rename(columns=rename)
    else:
        df=pd.read_csv(path)
    if "fog_flag" not in df.columns:
        return None
    need=COLS+["fog_flag"]
    df=df[need].apply(pd.to_numeric,error="coerce").dropna()
    return df

In [57]:
OBS=int(OBS_S*FS)
HOR=int(HORIZON_S*FS)
STRIDE=int(STRIDE_S*FS)


def listing_turning_files(root):
    files=[]
    for p in Path(root).rglob("SUB.*"):
        if "standing" in p.stem.lower():
            continue
        if p.suffix.lower() not in {".csv",".txt"}:
            continue
        files.append(p)

    by={}
    for p in files:
        by.setdefault(p.stem,[]).append(p)

    out=[]
    for stem,ps in by.items():
        txts=[p for p in ps if p.suffix.lower()==".txt"]
        csvs=[p for p in ps if p.suffix.lower()==".csv"]
        out.append(txts[0] if txts else csvs[0])
    return sorted(out)


import re

def subject_of(path):
    m=re.search(r"(SUB\d)",Path(path).stem,re.I)
    return m.group(1).upper() if m else "UNK"

import numpy as np
def make_windows(df):
    x=df[COLS].to_numpy(np.float32)
    y=df["fog_flags"].to_numpy(np.int64)
    Xs,ys=[],[]
    t=0
    while t+OBS+HOR<=len(df):
        Xs.append(x[t:t+OBS])
        future=y[t+OBS:t+OBS+HOR]
        ys.append(1 if future.mean()>=FOG_FRAC else 0)
        t+=STRIDE
    if not Xs:
        return None,None
    return np.stack(Xs),np.array(ys,dtype=np.int64)

In [59]:
print(list(ROOT.iterdir())[:30])
print("txt:", len(list(ROOT.glob("SUB*.txt"))))
print("csv:", len(list(ROOT.glob("SUB*.csv"))))

[PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB02_1.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB20_standing.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB21_standing.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB03_2.txt'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB34_standing.txt'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB01_2.txt'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB35_1.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB31_1.txt'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB11_1.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_data/IMU/SUB07_2.csv'), PosixPath('/home/workstation-p/Downloads/Projects/FYDP/dataset/imu_d

In [58]:
files=listing_turning_files(ROOT)
print("turning files",len(files))
rows=[]
X_all,y_all,subj_all=[],[],[]
for fp in files:
    df=read_trial(fp)

    if df is None or len(df)<OBS+HOR:
        continue
    X,y=make_windows(df)
    if X is None:
        continue
    sub=subject_of(fp)
    X_all.append(X)
    y_all.append(y)
    subj_all.append(np.array([sub]*len(y)))
    rows.append((fp.name,sub,len(y),y.mean()))

X=np.concatenate(X_all,axis=0)
y=np.concatenate(y_all,axis=0)
subj=np.concatenate(subj_all,axis=0)
print("windows:", X.shape, "FOG rate:", y.mean(), "subjects:", len(set(subj)))
pd.DataFrame(rows, columns=["file", "subject", "n_win", "fog_rate"]).head()

turning files 0


ValueError: need at least one array to concatenate